In [1]:
import sys
sys.path.append("../src")

In [2]:
import pandas as pd
from pathlib import Path
from modelado import (asignar_estaciones, calcular_grados, filtrar_entrenamiento,
                      calcular_temperatura_media, armar_futuro,
                      pronosticar_cuantiles, maximo_estacional,
                      preparar_serie_hw, pronosticar_hw,
                      calcular_maximos_estacionales, pronostico_ingenuo, fechas_estacion, pronostico_retrospectivo)

In [3]:
inicio = pd.Timestamp("2021-05-01")
fin = pd.Timestamp("2021-10-01")

In [4]:
ruta_serie = Path("../data/processed/SERIE_TRABAJO_CTRELEUW.csv")
serie = pd.read_csv(ruta_serie, parse_dates=["MES"])
serie = asignar_estaciones(serie)

In [5]:
pasado = serie[serie["MES"] < inicio].copy()
pasado.head()

,MES,MAX_REAL_MW,POTENCIA_DECLARADA_MW,EDICION,DESCARGA,REVISADO,POSIBLE_COPIA_DECLARACION,TEMPERATURA_MEDIA_C,TEMPERATURA_MAX_C,TEMPERATURA_MIN_C,ESTACION,ANIO_ESTACION
0,2019-01-01,44.221,44.426,2022-12,2026-10-13,False,False,21.974194,39.5,8.0,VERANO,2018
1,2019-02-01,45.960,44.798,2022-12,2026-10-13,False,False,21.007143,40.2,7.5,VERANO,2018
2,2019-03-01,43.629,43.833,2022-12,2026-10-13,False,False,18.341935,35.2,7.0,VERANO,2018
3,2019-04-01,43.363,43.567,2022-12,2026-10-13,False,False,16.016667,29.0,2.9,VERANO,2018
4,2019-05-01,44.244,44.427,2022-12,2026-10-13,False,False,11.293548,23.9,-1.7,INVIERNO,2019


In [6]:
pasado = calcular_grados(pasado)
pasado.head()

,MES,MAX_REAL_MW,POTENCIA_DECLARADA_MW,EDICION,DESCARGA,REVISADO,POSIBLE_COPIA_DECLARACION,TEMPERATURA_MEDIA_C,TEMPERATURA_MAX_C,TEMPERATURA_MIN_C,ESTACION,ANIO_ESTACION,GRADOS_FRIO,GRADOS_CALOR
0,2019-01-01,44.221,44.426,2022-12,2026-10-13,False,False,21.974194,39.5,8.0,VERANO,2018,0.000000,7.974194
1,2019-02-01,45.960,44.798,2022-12,2026-10-13,False,False,21.007143,40.2,7.5,VERANO,2018,0.000000,7.007143
2,2019-03-01,43.629,43.833,2022-12,2026-10-13,False,False,18.341935,35.2,7.0,VERANO,2018,0.000000,4.341935
3,2019-04-01,43.363,43.567,2022-12,2026-10-13,False,False,16.016667,29.0,2.9,VERANO,2018,0.000000,2.016667
4,2019-05-01,44.244,44.427,2022-12,2026-10-13,False,False,11.293548,23.9,-1.7,INVIERNO,2019,2.706452,0.000000


In [7]:
entrenamiento = filtrar_entrenamiento(pasado)
len(entrenamiento)

20

In [8]:
temp_media = calcular_temperatura_media(pasado)

In [9]:
futuro = armar_futuro(inicio, fin, temp_media)

In [10]:
tabla_ordenada = pronosticar_cuantiles(entrenamiento, futuro)

In [11]:
cuantiles_maximo = maximo_estacional(tabla_ordenada)
cuantiles_maximo

array([45.41468124, 46.0272731 , 47.26081405])

In [12]:
serie_hw = preparar_serie_hw(pasado)

In [13]:
pronostico_hw = pronosticar_hw(serie_hw, 6)

In [14]:
pico_hw = pronostico_hw.max()
pico_hw

np.float64(40.11439063510798)

In [15]:
invierno_2021 = fechas_estacion("INVIERNO", 2021)
verano_2025 = fechas_estacion("VERANO", 2025)
print("Invierno 2021: ",invierno_2021)
print("Verano 2025: ",verano_2025)

Invierno 2021:  (Timestamp('2021-05-01 00:00:00'), Timestamp('2021-10-01 00:00:00'))
Verano 2025:  (Timestamp('2025-11-01 00:00:00'), Timestamp('2026-04-01 00:00:00'))


In [16]:
maximos = calcular_maximos_estacionales(serie)
maximos = pronostico_ingenuo(maximos)

In [17]:
evaluado = (maximos["UTILIZABLE"] == True) & (maximos["ANIO_ESTACION"] >= 2021)
estaciones_evaluadas = maximos.loc[evaluado, ["ESTACION", "ANIO_ESTACION", "MAXIMO_MW", "PRONOSTICO_ANT"]]
estaciones_evaluadas = estaciones_evaluadas.reset_index(drop=True)
estaciones_evaluadas

,ESTACION,ANIO_ESTACION,MAXIMO_MW,PRONOSTICO_ANT
0,INVIERNO,2021,42.352,NaN
1,INVIERNO,2022,44.156,42.352
2,INVIERNO,2023,42.448,44.156
3,INVIERNO,2024,41.960,42.448
4,INVIERNO,2025,44.436,41.960
5,VERANO,2021,44.380,44.396
6,VERANO,2022,45.528,44.380
7,VERANO,2023,48.340,45.528
8,VERANO,2024,47.856,48.340
9,VERANO,2025,44.244,47.856


In [18]:
retrospectivo = pronostico_retrospectivo(serie, estaciones_evaluadas)
retrospectivo

c:\Users\Ramir\OneDrive\Escritorio\Archivos_TFG\TFG\notebooks\../src\modelado.py:132: IterationLimitWarning: Maximum number of iterations (5000) reached.
  modelo_entrenado = modelo_cuantil.fit(q=cuantil, max_iter=max_iter)


,ESTACION,ANIO_ESTACION,MAXIMO_MW,PRONOSTICO_ANT,Q10_MW,Q50_MW,Q90_MW,HW_MW
0,INVIERNO,2021,42.352,NaN,45.414681,46.027273,47.260814,40.114391
1,INVIERNO,2022,44.156,42.352,43.734627,45.302328,46.840297,42.563225
2,INVIERNO,2023,42.448,44.156,43.181230,45.017425,46.953116,42.098513
3,INVIERNO,2024,41.960,42.448,42.678967,44.313533,46.699444,42.394445
4,INVIERNO,2025,44.436,41.960,42.253762,43.863804,46.760655,42.071078
5,VERANO,2021,44.380,44.396,43.781919,44.340467,46.007473,41.569720
6,VERANO,2022,45.528,44.380,43.314707,44.314921,45.871326,40.944850
7,VERANO,2023,48.340,45.528,42.831921,44.551764,45.881774,42.461331
8,VERANO,2024,47.856,48.340,42.829853,45.065222,46.901773,42.200645
9,VERANO,2025,44.244,47.856,42.697954,45.233164,47.076460,43.833393


In [19]:
retrospectivo.to_csv(Path("../data/processed/RETROSPECTIVO_CTRELEUW.csv"), index=False)